EMNIST-ByMerge

In [ ]:
#  EMNIST ByMerge: 814,255 chars, 47 UNBALANCED classes
#  Classes: digits 0-9 + letters where visually similar cases are merged.
#  Merged pairs (uppercase kept): C/c, I/i, J/j, K/k, L/l, M/m, O/o,
#                                  P/p, S/s, U/u, V/v, W/w, X/x, Y/y, Z/z
#  Key challenge: severe class imbalance — macro F1 matters more than accuracy.
#  Horizontal flip ENABLED — mixed digit+letter classes, flip helps letters
#  but we accept minor digit confusion as trade-off for letter regularisation.

In [ ]:
# importing necessary dependencies
import os, random, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
import math

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

#  CONFIG
DATASET      = "EMNIST ByMerge"
NUM_CLASSES  = 47
IMG          = 28
BS           = 64
EPOCHS       = 100
LR           = 5e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.10
WARMUP_EP    = 5
VAL_SPLIT    = 0.10
RESULTS_DIR  = "./results/bymerge"
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LABELS = (
    [str(d) for d in range(10)]
    + ['A','B','C','D','E','F','G','H','I','J','K','L','M',
       'N','O','P','Q','R','S','T','U','V','W','X','Y','Z',
       'a','b','d','e','f','g','h','n','q','r','t']
)  # 47 labels

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"[INFO] Device: {DEVICE}")

#  TRANSFORMS
#  EMNIST images are transposed vs standard — apply fix via rotate+flip.
emnist_fix = transforms.Compose([
    transforms.Lambda(lambda img: transforms.functional.rotate(img, -90)),
    transforms.Lambda(lambda img: transforms.functional.hflip(img)),
])

train_transform = transforms.Compose([
    emnist_fix,
    transforms.RandomAffine(degrees=0, translate=(0.07, 0.07)),   # ~2px pad+crop equiv
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),                                          # [0,1]
    transforms.Normalize((0.5,), (0.5,)),                          # [-1,1]
    transforms.Lambda(lambda x: x + torch.randn_like(x) * 0.02),  # Gaussian noise
    transforms.Lambda(lambda x: x.clamp(-1.0, 1.0)),
])

val_transform = transforms.Compose([
    emnist_fix,
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

#  DATASETS
print(f"[INFO] Loading {DATASET} ...")
full_train = datasets.EMNIST(root="./data", split="bymerge", train=True,
                              download=True, transform=train_transform)
test_ds    = datasets.EMNIST(root="./data", split="bymerge", train=False,
                              download=True, transform=val_transform)

n_val   = int(len(full_train) * VAL_SPLIT)
n_train = len(full_train) - n_val
train_subset, val_subset = random_split(
    full_train, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED)
)
# val subset should use val_transform — wrap it
val_subset.dataset = datasets.EMNIST(root="./data", split="bymerge", train=True,
                                      download=False, transform=val_transform)

print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {len(test_ds):,}")

train_loader = DataLoader(train_subset, batch_size=BS, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_subset,   batch_size=BS, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,      batch_size=BS, shuffle=False,
                          num_workers=4, pin_memory=True)

#  CLASS WEIGHTS
print("[INFO] Computing class weights ...")
label_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for _, label in train_subset:
    label_counts[int(label)] += 1

total_samples = label_counts.sum()
class_weights = total_samples / (NUM_CLASSES * np.maximum(label_counts, 1))
class_weights = np.clip(class_weights, 0.1, 10.0)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
print(f"[INFO] Class weight range: [{class_weights.min():.2f}, {class_weights.max():.2f}]")

#  BUILDING BLOCKS
def gelu(x):
    return F.gelu(x)


class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Linear(channels, max(1, channels // reduction))
        self.fc2 = nn.Linear(max(1, channels // reduction), channels)

    def forward(self, x):
        b, c, _, _ = x.shape
        attn = self.gap(x).view(b, c)
        attn = F.relu(self.fc1(attn))
        attn = torch.sigmoid(self.fc2(attn)).view(b, c, 1, 1)
        return x * attn


class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)

    def forward(self, x):
        r = x
        x = F.gelu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return F.gelu(x + r)


class DenseResBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.proj = None
        if in_ch != out_ch:
            self.proj = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, bias=False),
                nn.BatchNorm2d(out_ch),
            )
        self.r1 = ResidualBlock(out_ch)
        self.r2 = ResidualBlock(out_ch)
        self.r3 = ResidualBlock(out_ch)
        # concat of 3 residual blocks → 3*out_ch → out_ch
        self.fuse = nn.Sequential(
            nn.Conv2d(3 * out_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        # Depthwise strided downsampling
        self.dw   = nn.Conv2d(out_ch, out_ch, 3, stride=2, padding=1,
                              groups=out_ch, bias=False)
        self.pw   = nn.Conv2d(out_ch, out_ch, 1, bias=False)
        self.bn   = nn.BatchNorm2d(out_ch)

    def forward(self, x):
        if self.proj is not None:
            x = F.gelu(self.proj(x))
        r1  = self.r1(x)
        r2  = self.r2(r1)
        r3  = self.r3(r2)
        cat = torch.cat([r1, r2, r3], dim=1)
        out = F.gelu(self.fuse(cat))
        out = F.gelu(self.bn(self.pw(self.dw(out))))
        return out


class AdaptiveFilterCapsule(nn.Module):
    """Lightweight capsule routing (capsule_dim=32)."""
    def __init__(self, in_features, num_classes, capsule_dim=32):
        super().__init__()
        self.num_classes = num_classes
        self.capsule_dim = capsule_dim
        self.fc1  = nn.Linear(in_features, 256)
        self.fc2  = nn.Linear(256, num_classes * capsule_dim)
        self.bn   = nn.BatchNorm1d(num_classes)

    def forward(self, x):
        h     = F.gelu(self.fc1(x))
        h     = self.fc2(h).view(-1, self.num_classes, self.capsule_dim)  # (B, K, D)
        x_exp = x.unsqueeze(1).expand(-1, self.num_classes, -1)           # (B, K, F)
        x_sl  = x_exp[:, :, :self.capsule_dim]                            # (B, K, D)
        caps  = (x_sl * h).sum(dim=-1)                                    # (B, K)
        return self.bn(caps)

#  MODEL
class WhatNetByMerge(nn.Module):
    def __init__(self, num_classes=47, image_size=28):
        super().__init__()
        K = num_classes

        # Stem
        self.stem_t = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32))
        self.stem_h = nn.Sequential(
            nn.Conv2d(1, 16, (1, 5), padding=(0, 2), bias=False), nn.BatchNorm2d(16))
        self.stem_v = nn.Sequential(
            nn.Conv2d(1, 16, (5, 1), padding=(2, 0), bias=False), nn.BatchNorm2d(16))
        self.stem_ca  = ChannelAttention(64)
        self.stem_proj = nn.Sequential(
            nn.Conv2d(64, 64, 1, bias=False), nn.BatchNorm2d(64))

        # Encoder
        self.enc1 = DenseResBlock(64,  64)
        self.enc2 = DenseResBlock(64,  128)
        self.enc3 = DenseResBlock(128, 256)

        #  Multi-scale GAP projections
        self.gap_proj1 = nn.Linear(64,  128)
        self.gap_proj2 = nn.Linear(128, 128)
        # enc3 goes directly (256 features); fused = 128+128+256 = 512

        #  Capsule
        self.afc = AdaptiveFilterCapsule(512, K, capsule_dim=32)

        #  Dense head
        self.head_dense  = nn.Linear(512, 256, bias=False)
        self.head_ln     = nn.LayerNorm(256)
        self.head_drop   = nn.Dropout(0.25)
        self.head_logits = nn.Linear(256, K)

        #  Gate
        self.gate = nn.Linear(2 * K, 2)

    def forward(self, x):
        # Stem
        t    = F.gelu(self.stem_t(x))
        h    = F.gelu(self.stem_h(x))
        v    = F.gelu(self.stem_v(x))
        stem = torch.cat([t, h, v], dim=1)
        stem = self.stem_ca(stem)
        stem = F.gelu(self.stem_proj(stem))

        # Encoder
        e1 = self.enc1(stem)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)

        # Multi-scale GAP
        g1 = F.gelu(self.gap_proj1(e1.mean(dim=[2, 3])))
        g2 = F.gelu(self.gap_proj2(e2.mean(dim=[2, 3])))
        g3 = e3.mean(dim=[2, 3])
        fused = torch.cat([g1, g2, g3], dim=1)  # (B, 512)

        # Capsule
        afc_out = self.afc(fused)

        # Dense head
        x = F.gelu(self.head_ln(self.head_dense(fused)))
        x = self.head_drop(x)
        x = self.head_logits(x)

        # Gated fusion
        gate_w   = F.softmax(self.gate(torch.cat([x, afc_out], dim=1)), dim=1)
        x_scaled = x       * gate_w[:, 0:1]
        a_scaled = afc_out * gate_w[:, 1:2]
        return x_scaled + a_scaled

#  LR SCHEDULE  (warmup + cosine decay)
def make_warmup_cosine_scheduler(optimizer, total_steps, warmup_steps):
    def lr_lambda(current_step):
        if current_step < warmup_steps:
            return float(current_step) / max(warmup_steps, 1)
        progress = (current_step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return max(0.5 * (1.0 + math.cos(math.pi * progress)), 1e-6 / LR)
    return LambdaLR(optimizer, lr_lambda)


#  LOSS  (label smoothing + per-sample class weighting)
class WeightedLabelSmoothingLoss(nn.Module):
    def __init__(self, num_classes, smoothing, class_weights):
        super().__init__()
        self.smoothing      = smoothing
        self.num_classes    = num_classes
        self.class_weights  = class_weights  # (K,) tensor on device

    def forward(self, logits, targets):
        log_probs  = F.log_softmax(logits, dim=-1)
        smooth_val = self.smoothing / self.num_classes
        # Build smooth target distribution
        with torch.no_grad():
            smooth_targets = torch.full_like(log_probs, smooth_val)
            smooth_targets.scatter_(1, targets.unsqueeze(1),
                                    1.0 - self.smoothing + smooth_val)
        loss        = -(smooth_targets * log_probs).sum(dim=-1)          # (B,)
        sample_wts  = self.class_weights[targets]                        # (B,)
        return (loss * sample_wts).mean()

#  TRAIN
model = WhatNetByMerge(NUM_CLASSES, IMG).to(DEVICE)
print(f"[INFO] Params: {sum(p.numel() for p in model.parameters()):,}")

steps_per_epoch = len(train_loader)
total_steps     = steps_per_epoch * EPOCHS
warmup_steps    = steps_per_epoch * WARMUP_EP

optimizer   = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler   = make_warmup_cosine_scheduler(optimizer, total_steps, warmup_steps)
criterion   = WeightedLabelSmoothingLoss(NUM_CLASSES, LABEL_SMOOTH, class_weights_tensor)

best_val_acc  = 0.0
best_state    = None
patience      = 12
no_improve    = 0
history       = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}

print(f"[INFO] Steps/epoch: {steps_per_epoch} | Total: {total_steps}")

for epoch in range(1, EPOCHS + 1):
    # Training
    model.train()
    run_loss, run_correct, run_total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        scheduler.step()

        run_loss    += loss.item() * images.size(0)
        run_correct += (logits.argmax(1) == labels).sum().item()
        run_total   += images.size(0)

    train_loss = run_loss / run_total
    train_acc  = run_correct / run_total

    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits  = model(images)
            loss    = criterion(logits, labels)
            val_loss    += loss.item() * images.size(0)
            val_correct += (logits.argmax(1) == labels).sum().item()
            val_total   += images.size(0)

    val_loss /= val_total
    val_acc   = val_correct / val_total

    history["loss"].append(train_loss)
    history["accuracy"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_acc)

    print(f"Epoch {epoch:>3}/{EPOCHS} | "
          f"loss {train_loss:.4f}  acc {train_acc*100:.2f}% | "
          f"val_loss {val_loss:.4f}  val_acc {val_acc*100:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        torch.save(best_state, os.path.join(RESULTS_DIR, "best_model.pt"))
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"[INFO] Early stopping at epoch {epoch}")
            break

# Restore best weights
if best_state is not None:
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

#  EVALUATE
model.eval()
tp = np.zeros(NUM_CLASSES); fp = np.zeros(NUM_CLASSES); fn = np.zeros(NUM_CLASSES)
correct_per_class = np.zeros(NUM_CLASSES); total_per_class = np.zeros(NUM_CLASSES)
test_loss_total, test_total = 0.0, 0

ce_loss = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=LABEL_SMOOTH)

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        logits = model(images)
        loss   = ce_loss(logits, labels)
        test_loss_total += loss.item() * images.size(0)
        test_total      += images.size(0)

        preds = logits.argmax(1).cpu().numpy()
        lbls  = labels.cpu().numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
            correct_per_class[c] += np.sum(preds[lbls == c] == c)
            total_per_class[c]   += np.sum(lbls == c)

test_loss     = test_loss_total / test_total
test_acc      = correct_per_class.sum() / total_per_class.sum() * 100.0
prec          = tp / (tp + fp + 1e-8)
rec           = tp / (tp + fn + 1e-8)
f1            = 2 * prec * rec / (prec + rec + 1e-8)
macro_f1      = float(f1.mean() * 100.0)
per_class_acc = correct_per_class / np.maximum(total_per_class, 1) * 100.0
worst5_idx    = np.argsort(per_class_acc)[:5]

print(f"\n[RESULT] Test Acc : {test_acc:.2f}%")
print(f"[RESULT] Macro F1 : {macro_f1:.2f}%")
print(f"[RESULT] Test Loss: {test_loss:.4f}")
print(f"[RESULT] Params   : {sum(p.numel() for p in model.parameters()):,}")
print(f"\n[RESULT] Worst-5 classes:")
for idx in worst5_idx:
    print(f"         '{LABELS[idx]}' (cls {idx:>2}) = {per_class_acc[idx]:.1f}%")

results = {
    "dataset": "EMNIST_BYMERGE", "num_classes": NUM_CLASSES,
    "test_acc": round(float(test_acc), 2), "macro_f1": round(macro_f1, 2),
    "test_loss": round(float(test_loss), 4),
    "params": sum(p.numel() for p in model.parameters()),
    "per_class_acc": {LABELS[c]: round(float(per_class_acc[c]), 2) for c in range(NUM_CLASSES)},
}
with open(os.path.join(RESULTS_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2)
with open(os.path.join(RESULTS_DIR, "history.json"), "w") as f:
    json.dump({k: [float(v) for v in vs] for k, vs in history.items()}, f, indent=2)

print(f"\n[INFO] Saved to {RESULTS_DIR}/")
print("[DONE]")

[INFO] Device: cuda
[INFO] Loading EMNIST ByMerge ...


100%|██████████| 562M/562M [00:03<00:00, 181MB/s]  


[INFO] Train: 628,139 | Val: 69,793 | Test: 116,323
[INFO] Computing class weights ...
[INFO] Class weight range: [0.39, 5.86]
[INFO] Params: 5,735,419
[INFO] Steps/epoch: 9815 | Total: 981500
Epoch   1/100 | loss 1.7377  acc 65.40% | val_loss 1.1357  val_acc 84.93%
Epoch   2/100 | loss 1.1332  acc 82.97% | val_loss 1.0974  val_acc 83.29%
Epoch   3/100 | loss 1.0645  acc 84.74% | val_loss 1.0338  val_acc 84.74%
Epoch   4/100 | loss 1.0289  acc 85.59% | val_loss 1.0086  val_acc 85.29%
Epoch   5/100 | loss 1.0071  acc 86.16% | val_loss 0.9980  val_acc 86.01%
Epoch   6/100 | loss 0.9862  acc 86.75% | val_loss 0.9724  val_acc 88.30%
Epoch   7/100 | loss 0.9690  acc 87.30% | val_loss 0.9683  val_acc 87.28%
Epoch   8/100 | loss 0.9563  acc 87.68% | val_loss 0.9514  val_acc 88.42%
Epoch   9/100 | loss 0.9472  acc 87.94% | val_loss 0.9443  val_acc 88.93%
Epoch  10/100 | loss 0.9397  acc 88.16% | val_loss 0.9436  val_acc 88.37%
Epoch  11/100 | loss 0.9332  acc 88.36% | val_loss 0.9386  val_acc 